# §37 — State katkısı biriken geçmişle nasıl değişiyor?

**Otorite ön-kayıt `RESULTS.md` §37'de.** Bu markdown kolaylık kopyasıdır.

§36 katkıyı **tek bir mesafede** ölçtü: exp +0.0350, cubic −0.2817 nat/token.
Ölçmediği şey, katkının biriken geçmişle nasıl davrandığı.

**Δ(D) = CE(state ölçüm chunk'ında sıfır) − CE(state D chunk geriden taşınıyor)**
D ∈ {1, 2, 4, 8, 16}

**Neden Faz A'dan önce:** cevap Faz A'nın hangi problemi çözeceğini belirliyor.
- **DOYUYOR** → kanal dolu, kısıt **kapasite**; seçici yazım yanlış kaldıraç.
- **BÜYÜYOR** → kanal kararlı ve az kullanılmış, kısıt **ne yazıldığı**;
  seçici yazım doğru kaldıraç.
- **AZALIYOR** → birikim exp için de zararlı; "daha iyi içerik yaz" yönü kapanır.

**D=1 iç tutarlılık kontrolüdür:** §36'nın koşulunu tekrar üretir ve
+0.0350 / −0.2817 civarına düşmelidir. Düşmezse iki koşu arasında bir şey
farklı demektir ve tarama kıyaslanabilir değildir.

**Test önceden Wilcoxon.** Eşik 0.02 nat/token. Güç kapısı: exp'te Δ(1) > 0,
p < 0.05 — yoksa menzili ölçülecek bir katkı yok, **sonuçsuz** yazılır.

**Ölçmediği:** Δ(D), *D chunk geçmişe sahip olmanın* değeri; *tam D chunk
önceki içeriğin* değeri değil. Menzil ve birikim farklı büyüklükler; bu deney
birikimi ölçüyor.


In [ ]:
# --- 1. KURULUM ---
import os, sys, glob, json, subprocess, math, time
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'GPU YOK. Accelerator > T4 sec.'
OUT = os.path.join(BASE,'v37'); os.makedirs(OUT, exist_ok=True)
print('repo:', REPO, '| cihaz:', torch.cuda.get_device_name(0), '| cikti:', OUT)

In [ ]:
# --- 2. IKIZ CHECKPOINTLERI BUL + PARMAK IZIYLE DOGRULA + KATMAN KUMESI ---
# NOT: checkpoint'ler .gitignore'da (*.pt, checkpoints/) -> repo klonunda YOKLAR.
# Kaggle: Add Data > Your Datasets ile iki .pt dosyasini yukle (bkz. markdown).
#
# DOSYA ADINA GUVENILMEZ. KAYNAK.md'deki ders: Run 1'in 'hfp_graft_final.pt'si
# bir ara 'run5' etiketiyle dolasti. Kimlik PARMAK IZIYLE dogrulanir:
#   Run 1  -> out_gain ort ~0.75   (YANLIS dosya)
#   Run 5  -> out_gain ort ~0.237  (cubic ikizi)
#   Run 6  -> out_gain ort ~0.239  (exp ikizi)
import re
ROOTS = ['/kaggle/input', REPO, BASE, '/content/drive/MyDrive', '/content', os.path.expanduser('~')]
cands = []
for r in ROOTS:
    if r and os.path.isdir(r): cands += glob.glob(f'{r}/**/*.pt', recursive=True)
cands = sorted(set(cands))
assert cands, ('HIC .pt DOSYASI YOK.\n'
               '  Kaggle: Add Data > Upload > iki dosyayi yukle:\n'
               '    checkpoints/graft_run5/hfp_graft_final.pt      (cubic)\n'
               '    checkpoints/graft_run6_exp/hfp_graft_exp_final.pt (exp)\n'
               '  Bunlar .gitignore\'da oldugu icin repo klonunda YOK.')

def probe(path):
    sd = torch.load(path, map_location='cpu')
    if isinstance(sd, dict) and 'm' in sd and isinstance(sd['m'], dict): sd = sd['m']
    og = [v.flatten() for k, v in sd.items() if k.endswith('out_gain')]
    if not og: return None
    og = torch.cat(og)
    ls = sorted({int(m.group(1)) for k in sd
                 for m in [re.search(r'layers\.(\d+)\.self_attn', k)] if m})
    return {'sd': sd, 'n': len(sd), 'og': float(og.mean()), 'ogsd': float(og.std()), 'layers': ls}

print(f"{'out_gain':>9} {'std':>7} {'tensor':>7} {'kat':>4}  dosya")
print('-'*84)
info = {}
for c in cands:
    p = probe(c)
    if p is None: continue
    info[c] = p
    flag = '  <-- Run1? (out_gain ~0.75, YANLIS)' if p['og'] > 0.5 else ''
    print(f"{p['og']:>9.4f} {p['ogsd']:>7.4f} {p['n']:>7} {len(p['layers']):>4}  {c}{flag}")
assert info, 'Bulunan .pt dosyalarinin hicbiri graft checkpointi degil (out_gain yok).'

# Secim: exp = dosya adinda 'exp'; cubic = 'exp' YOK. Ikisinde de parmak izi kapisi.
def pick(arm):
    want_exp = (arm == 'exp')
    hits = [c for c, p in info.items()
            if (('exp' in os.path.basename(c).lower()) == want_exp)
            and 'final' in os.path.basename(c).lower()
            and 0.15 <= p['og'] <= 0.40]          # PARMAK IZI KAPISI: Run1'i (~0.75) eler
    assert hits, (f'{arm} ikizi bulunamadi. Ya dosya yok, ya parmak izi disinda '
                  f'(0.15-0.40 bandi). Yukaridaki tabloya bak: Run1 (~0.75) KABUL EDILMEZ.')
    assert len(hits) == 1, f'{arm} icin BIRDEN FAZLA aday: {hits}. Fazlasini kaldir.'
    return hits[0]

CKPT = {a: pick(a) for a in ('cubic', 'exp')}
SD_CUBIC, SD_EXP = info[CKPT['cubic']]['sd'], info[CKPT['exp']]['sd']
L_CUBIC, L_EXP   = info[CKPT['cubic']]['layers'], info[CKPT['exp']]['layers']
print()
for a in ('cubic', 'exp'):
    i = info[CKPT[a]]
    print(f'{a:>6}: out_gain {i["og"]:.4f} | tensor {i["n"]} | katman {len(i["layers"])} | {CKPT[a]}')
assert L_CUBIC == L_EXP, ('IKIZ DEGILLER: katman kumeleri farkli -> tek-degisken '
                          f'kosulu ihlal, deney KOSULAMAZ.\n  cubic {L_CUBIC}\n  exp   {L_EXP}')
GRAFT_LAYERS = L_CUBIC
print(f'\nortak katman kumesi ({len(GRAFT_LAYERS)}): {GRAFT_LAYERS}')
print('KAYNAK.md referansi: Run5 out_gain ~0.237 (cubic), Run6 ~0.239 (exp)')


In [ ]:
# --- 3. BASE MODEL + GRAFT KURUCU (yukleme DOGRULAMASI zorunlu, §30 hucre 7) ---
import glob, os
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode,
                                 enable_streaming, reset_streaming, HFPGraftAttention)
ROOTS = ['/kaggle/input', REPO, BASE, os.path.expanduser('~'), '.']
def find_one(pat):
    hits = []
    for r in ROOTS:
        if r and os.path.isdir(r): hits += glob.glob(f'{r}/**/{pat}', recursive=True)
    return max(set(hits), key=os.path.getmtime) if hits else None

# Qwen: config.json ILE AGIRLIGIN AYNI KLASORDE oldugu yeri ara.
# Not: repodaki hf_upload/hf_release/config.json agirlik icermiyor; sadece
# 'config.json' aramak onu bulup SRC'yi bos birakiyordu.
_cands = []
for _r in ROOTS:
    if not (_r and os.path.isdir(_r)): continue
    for _cfg in glob.glob(f'{_r}/**/config.json', recursive=True):
        _d = os.path.dirname(_cfg)
        if glob.glob(f'{_d}/*.safetensors') or glob.glob(f'{_d}/*.bin'):
            _cands.append(_d)
assert _cands, ('QWEN YOK. Kaggle sag panel > Add Input > Models > Qwen2.5-1.5B.\n'
                '  (Datasets sekmesi degil, Models sekmesi.)')
SRC = _cands[0]
tok = AutoTokenizer.from_pretrained(SRC)
print('base model:', SRC)


def build(arm):
    """arm in {'cubic','exp'} -> yuklenmis, DOGRULANMIS grafted model."""
    mode = 'cubic_flux_chunked' if arm == 'cubic' else 'exp'
    m = AutoModelForCausalLM.from_pretrained(SRC, torch_dtype=torch.float32).to(DEV).eval()
    graft_llama(m, GraftConfig(decay_mode=mode, write_rule='hybrid',
                               key_feature_map='dpfp', rec_block=16), layers=GRAFT_LAYERS)
    for mm in m.modules():
        if isinstance(mm, HFPGraftAttention): mm.out_gain.data.fill_(0.1)
    sd = SD_CUBIC if arm == 'cubic' else SD_EXP
    m.load_state_dict(sd, strict=False)
    set_graft_mode(m, 'student'); m.config.use_cache = True
    # --- YUKLEME DOGRULAMASI (strict=False sessizce hicbir sey yuklemeyebilir) ---
    own = dict(m.state_dict())
    matched = [k for k in sd if k in own and own[k].shape == sd[k].shape]
    missing = [k for k in sd if k not in own]
    same = sum(1 for k in matched if torch.equal(own[k].to(DEV), sd[k].to(DEV)))
    og = torch.cat([mm.out_gain.detach().flatten() for mm in m.modules()
                    if isinstance(mm, HFPGraftAttention)])
    ok = (len(matched) == len(sd)) and (same == len(matched)) and (og.std() > 1e-4)
    print(f'[{arm}] tensor {len(sd)} | eslesen {len(matched)} | bit-bit ayni {same} | '
          f'eksik isim {len(missing)} | out_gain ort {og.mean():.4f} std {og.std():.4f}')
    assert ok, (f'[{arm}] CHECKPOINT YUKLENMEDI. out_gain std ~0 ve ort ~0.1 ise '
                f'agirliklar EGITIMSIZ -> hicbir sayi raporlanamaz. eksik: {missing[:3]}')
    print(f'[{arm}] YUKLEME DOGRULANDI')
    return m
print('\nkurucu hazir (build("cubic") / build("exp"))')

In [ ]:
# --- 4. DEGERLENDIRME METNI (held-out) ---
# S1 distilasyonu WikiText-103 TRAIN uzerinde yapildi -> VALIDATION held-out'tur.
CHUNK, HEAD, N_CHUNK = 256, 32, 120   # chunk uzunlugu | endpoint penceresi | chunk sayisi
try:
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='validation')
    text = '\n\n'.join(t for t in ds['text'] if len(t.strip()) > 200)
    kaynak = 'wikitext-103-raw-v1/validation'
except Exception as e:
    p = find_one('wiki.valid.tokens') or find_one('wiki.valid.raw')
    assert p, f'Degerlendirme metni yok ({type(e).__name__}). WikiText-103 valid ekle.'
    text = open(p, encoding='utf-8').read(); kaynak = p
ids = tok(text, return_tensors='pt').input_ids[0]
need = CHUNK * (N_CHUNK + 1)
assert ids.numel() >= need, f'metin kisa: {ids.numel()} < {need}'
ids = ids[:need]
print(f'kaynak: {kaynak}\ntoken: {ids.numel():,} | chunk {CHUNK} x {N_CHUNK+1} | endpoint = ilk {HEAD} token')

In [ ]:
# --- 5. KOSU: D in {1,2,4,8,16}, iki kol ---
import torch.nn.functional as F
DS = [1, 2, 4, 8, 16]

@torch.no_grad()
def run_D(m, D):
    """Her olcum chunk'i icin: state D chunk geriden tasinmis hali vs sifir hali.
    Cache HER chunk'ta sifir; tek degisken state'in ne kadar geriden geldigi."""
    enable_streaming(m, True)
    carried, reset = [], []
    for t in range(D, N_CHUNK + 1):
        # --- TASINAN: t-D'den t'ye kadar state tasinir, cache her chunk sifir ---
        reset_streaming(m)
        for c in range(t - D, t):
            xf = ids[c*CHUNK:(c+1)*CHUNK].unsqueeze(0).to(DEV)
            m(xf, past_key_values=DynamicCache(), use_cache=True)
        x = ids[t*CHUNK:(t+1)*CHUNK].unsqueeze(0).to(DEV)
        lg = m(x, past_key_values=DynamicCache(), use_cache=True).logits
        carried.append(F.cross_entropy(lg[0,:HEAD,:], x[0,1:HEAD+1]).item())
        # --- SIFIR: ayni chunk, state yok ---
        reset_streaming(m)
        lg = m(x, past_key_values=DynamicCache(), use_cache=True).logits
        reset.append(F.cross_entropy(lg[0,:HEAD,:], x[0,1:HEAD+1]).item())
    return carried, reset

RES = {}
for arm in ('cubic', 'exp'):
    t0 = time.time(); m = build(arm); RES[arm] = {}
    for D in DS:
        c, r = run_D(m, D)
        RES[arm][D] = {'carried': c, 'reset': r}
        d = [ri - ci for ri, ci in zip(r, c)]
        print(f'[{arm}] D={D:>2}  D_ort = {sum(d)/len(d):+.4f} nat/token  (n={len(d)})', flush=True)
    del m; torch.cuda.empty_cache()
    print(f'[{arm}] bitti ({time.time()-t0:.0f}s)\n')
json.dump({'chunk':CHUNK,'head':HEAD,'n_chunk':N_CHUNK,'DS':DS,'kaynak':kaynak,
           'layers':GRAFT_LAYERS,'res':RES}, open(f'{OUT}/v37_raw.json','w'), indent=2)
print('ham veri:', f'{OUT}/v37_raw.json')

In [ ]:
# --- 6. GUC KAPISI + ON-KAYITLI HUKUM (§37) ---
import statistics as st, math
def wilcoxon(d):
    dd = [x for x in d if x != 0]; n = len(dd)
    if n < 6: return float('nan'), float('nan')
    order = sorted(range(n), key=lambda i: abs(dd[i])); rank = [0.0]*n
    i = 0
    while i < n:
        j = i
        while j+1 < n and abs(dd[order[j+1]]) == abs(dd[order[i]]): j += 1
        avg = (i+j)/2 + 1
        for k in range(i, j+1): rank[order[k]] = avg
        i = j + 1
    W = sum(rank[i] for i in range(n) if dd[i] > 0)
    z = (W - n*(n+1)/4) / math.sqrt(n*(n+1)*(2*n+1)/24)
    return z, math.erfc(abs(z)/math.sqrt(2))

D_ = {a: {D: [r-c for r, c in zip(RES[a][D]['reset'], RES[a][D]['carried'])]
          for D in DS} for a in ('cubic','exp')}

print('=== Δ(D) tablosu (nat/token; POZITIF = state yardim ediyor) ===')
print(f"{'D':>4} {'exp':>22} {'cubic':>22}")
for D in DS:
    row = ''
    for a in ('exp','cubic'):
        m = st.mean(D_[a][D]); z, p = wilcoxon(D_[a][D])
        row += f'  {m:+.4f} (p={p:.3g}, n={len(D_[a][D])})'.rjust(24)
    print(f'{D:>4}{row}')

print('\n=== IC TUTARLILIK: D=1, §36 ile ayni olmali ===')
print(f"  exp   D=1 = {st.mean(D_['exp'][1]):+.4f}   §36: +0.0350")
print(f"  cubic D=1 = {st.mean(D_['cubic'][1]):+.4f}   §36: -0.2817")
print('  (belirgin sapma varsa tarama §36 ile KIYASLANAMAZ -- once bunu cozun)')

z1, p1 = wilcoxon(D_['exp'][1]); m1 = st.mean(D_['exp'][1])
GATE = (m1 > 0) and (p1 < 0.05)
print(f'\n=== GUC KAPISI: exp D=1 = {m1:+.4f}, p={p1:.4g} -> '
      f'{"GECILDI" if GATE else "GECILMEDI"} ===')

print('\n=== ON-KAYITLI HUKUM (§37, birincil kol = exp) ===')
if not GATE:
    print('  SONUCSUZ (null DEGIL). En kisa mesafede bile olculebilir katki yok;')
    print('  menzili olculecek bir sey yok. Faz A bu girdi olmadan ilerler ve')
    print('  secici-yazim hipotezi cozulmemis kalir.')
else:
    # eslesmis fark: ayni chunk indekslerinde D=16 vs D=2
    n = min(len(D_['exp'][16]), len(D_['exp'][2]))
    off = len(D_['exp'][2]) - n            # D=2 daha uzun; kuyruklari hizala
    diff = [D_['exp'][16][i] - D_['exp'][2][i+off] for i in range(n)]
    md = st.mean(diff); z, p = wilcoxon(diff)
    print(f'  Δ(16) - Δ(2) = {md:+.4f} nat/token  (n={n}, Wilcoxon p={p:.4g})')
    print(f'  esik: 0.02 nat/token')
    if md >= 0.02 and p < 0.05:
        print('\n  => BUYUYOR: uzun gecmis yardim etmeye devam ediyor. Kanal kararli')
        print('     ve az kullanilmis -> Faz A\'nin kisiti NE YAZILDIGI.')
        print('     Secici yazim gerekcelendirilmis kaldiractir.')
    elif -md >= 0.02 and p < 0.05:
        print('\n  => AZALIYOR: birikim exp icin de net zararli. Girisim her iki')
        print('     retention yasasinda da baskin -> "daha iyi icerik yaz" yonu KAPANIR.')
    else:
        print('\n  => DOYUYOR: ~2 chunk otesi olculebilir bir sey eklemiyor.')
        print('     Kanal dolu -> Faz A\'nin kisiti KAPASITE; secici yazim tek basina')
        print('     beklenen kaldirac degil.')

print('\n--- cubic kolu (hukum YOK, §36 zaten kapatti; birikimle derinlesiyor mu) ---')
print('  ' + '  '.join(f'D={D}: {st.mean(D_["cubic"][D]):+.4f}' for D in DS))
print('  (§27a girisim aciklamasi derinlesme ONGORUYOR; teyit ederse mekanistik')
print('   dogrulama olur, yeni iddia degil)')